# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step. 
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file. 
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:** 

After some thinkering with available models on pytorch the backbone of Efficient Net V2 as selected, the last Fully Connected Layer used to generate the classification on the original model as repourpused as the feature vector generator. A embedding layer as created to convert features from the image and text to extract semantical information from features. To finish the labels predictions a LSTM Layer was used to hold the last predicted tokens on memory and a Fully Conected Layer takes the LSTM output and predicts the current token. Diagram for each component on Appendix II.


- `batch_size` - It was empirically adjusted to 64, it was the highest batch size that would fit on the GPU memory while allowing training with random input sizes from 348x348 to 512x512; 
- `vocab_threshold` - It was selected as 20 to allow a good dictionary size (5359 words) making the sentences more natural;
- `vocab_from_file` - During training this variable is defined as True, while during evaluating and testing it is defined as False;
- `embed_size` - After some experiments it was selected the value of 512;
- `hidden_size` - After some experiments it was selected the value of 512;
- `num_epochs` - This parameter was changed to `total_steps`. Due to the random selection of sentence size and then the random selection of the samples that has this same description sentence size it was not possible to control if all samples were visited to form an epoch, so it was more fitting to measure the training progressing in steps. During training 100.000 steps were used, which took 2 days;
- `save_every` - a intermediary model was saved every 10.000 steps;
- `print_every` - To log model training metrics MLFlow was used, training loss was logged every 100 steps, while evaluation metrics were logged every 10.000 steps;
- `log_file` - To hold the logs MLFlow was used.

### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and 
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:** 

After loading the image from file we apply a letterbox transformation on the image so it is padded with zeros so it has the same width and height while keeping the aspect ration from the original image, this numpy matrix is then converted into a Pytorch Tensor and the `RandomResize` transformation is used to generate samples in different scales. 

This process proved less ambiguous then a random crop sampling because it was not possible to control what portion on the scene each crop would sample, which might not have all the information described by the output caption and since the Coco Dataset has a lot of samples, data augumentation steps were not impactful, training on multiple scales though improved the capacity of extrating features on multiple aspects of the same scene.

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters()) 
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:** 

On the encoder all layers extracted from Efficient Net V2 were not trainable and the weights trained on IMAGENET dataset was used. This is decision was taken to make training of a bigger batch possible otherwise it would be limited to a batch size of 8 which would lead to a noisy update on the weights. 

As trainable parameters the final convolution layer of the decoder and the LSTM and Fully Connected layers of the decoder were used.

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:** 

Initially the Adam optimizer was able to make the model converge into a resonable result while SGD always diverged, even after altering the momentum and dampening, but after reading [You Only Look Once: Unified, Real-Time Object Detection](https://pjreddie.com/media/files/papers/yolo_1.pdf) a soft start on the learning as described on the article helped to stabilize the weights update and make the model converge to good results. 

Using this soft start the optimizers gave close results so it was kept the Adam optimizer. Multiple learning rate functions were also experimented and for the final result the multi step function with soft start was used.

In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
import sys
# sys.path.append('/opt/cocoapi/PythonAPI')
from pycocotools.coco import COCO
from data_loader import get_loader
from model import EncoderCNN, DecoderRNN
import math


## TODO #1: Select appropriate values for the Python variables below.
batch_size = 20          # batch size
vocab_threshold = 5        # minimum word count threshold
vocab_from_file = True    # if True, load existing vocab file
embed_size = 256           # dimensionality of image and word embeddings
hidden_size = 512          # number of features in hidden state of the RNN decoder
num_epochs = 3             # number of training epochs
save_every = 1             # determines frequency of saving model weights
print_every = 100          # determines window for printing average loss
log_file = 'training_log.txt'       # name of file with saved training loss and perplexity

# (Optional) TODO #2: Amend the image transform below.
transform_train = transforms.Compose([ 
    transforms.ToTensor(),                           # convert the PIL Image to a tensor
    transforms.Resize(256),                          # smaller edge of image resized to 256
    transforms.RandomCrop(224),                      # get 224x224 crop from random location
    transforms.RandomHorizontalFlip(),               # horizontally flip image with probability=0.5
    transforms.Normalize((0.485, 0.456, 0.406),      # normalize image for pre-trained model
                         (0.229, 0.224, 0.225))])

# Build data loader.
data_loader = get_loader(transform=transform_train,
                         mode='train',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=vocab_from_file)

# The size of the vocabulary.
vocab_size = len(data_loader.dataset.vocab)

# Initialize the encoder and decoder. 
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size)

# Move models to GPU if CUDA is available. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder.to(device)

# Define the loss function. 
criterion = nn.CrossEntropyLoss().cuda() if torch.cuda.is_available() else nn.CrossEntropyLoss()

# TODO #3: Specify the learnable parameters of the model.
params = list(encoder.parameters())+list(decoder.parameters())

# TODO #4: Define the optimizer.
optimizer = torch.optim.Adam(params,1e-3)

# Set the total number of training steps per epoch.
total_step = math.ceil(len(data_loader.dataset.caption_lengths) / data_loader.batch_sampler.batch_size)

Vocabulary successfully loaded from vocab.pkl file!
loading annotations into memory...
Done (t=0.85s)
creating index...
index created!
Obtaining caption lengths...


100%|██████████| 414113/414113 [00:47<00:00, 8792.86it/s]
/usr/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/usr/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

In [ ]:
import time
import math
import numpy as np

import torch
import torch.nn as nn
import torch.utils.data as data
from torchvision import transforms
from torchmetrics import Accuracy, JaccardIndex
import tqdm

from data_loader import get_loader
from model import ImageCaptioner, get_transform, get_inference_transform
import mlflow

import schedulers


## TODO #1: Select appropriate values for the Python variables below.
batch_size = 64          # batch size
vocab_threshold = 20        # minimum word count threshold
vocab_from_file = False    # if True, load existing vocab file

lr = 1e-3
last_every = 100
opt_name = "adam"
scheduler_name = "steps"
dropout = 0.4
backbone_type = "efficientnet_v2"

transform_train = get_transform()

# Build data loader.
data_loader = get_loader(mode='train',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=vocab_from_file,
                         num_workers=8)

transform_valid = get_inference_transform()
data_loader_valid = get_loader(mode='valid',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=True,
                         num_workers=8)


save_every = 10000

# The size of the vocabulary.
vocab_size = len(data_loader.dataset.vocab)
embed_size = 512          # dimensionality of image and word embeddings
hidden_size = 512         # number of features in hidden state of the RNN decoder
num_layers = 1
total_step = 100000
rampup_period = 1000
training_params = {"opt":opt_name,
                   "scheduler":scheduler_name, 
                   "num_layers":num_layers, 
                   "lr":lr, 
                   "batch_size":batch_size, 
                   "vocab_threshold":vocab_threshold, 
                   "embed_size":embed_size, 
                   "hidden_size":hidden_size, 
                   "dropout": dropout,
                   "steps":total_step, 
                   "vocab_size":vocab_size,
                   "backbone_type":backbone_type,
                   "rampup_period":rampup_period}

# Initialize the encoder and decoder. 
model = ImageCaptioner(backbone_type,embed_size, hidden_size, vocab_size, num_layers, dropout=dropout, max_len=max(data_loader.dataset.caption_lengths))
model.train()

# Move models to GPU if CUDA is available. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load the trained weights.
# encoder.load_state_dict(torch.load(os.path.join('./models/encoder-4.pkl'),map_location=device))
# decoder.load_state_dict(torch.load(os.path.join('./models/decoder-4.pkl'),map_location=device))

# Define the loss function. 
criterion = nn.CrossEntropyLoss()
# criterion = nn.MultiLabelSoftMarginLoss()
# eval_criterion = Accuracy(task="multiclass", num_classes=vocab_size)
eval_criterion = JaccardIndex(task="multiclass", num_classes=vocab_size)

if torch.cuda.is_available():
    criterion = criterion.cuda()
    eval_criterion = eval_criterion.cuda()

# TODO #3: Specify the learnable parameters of the model.
# params = list(model.encoder.parameters())+list(model.decoder.parameters())
params = model.parameters()


# Set the total number of training steps per epoch.

# total_step = 2000

mlflow.set_tracking_uri("http://mlflow.cluster.local")
experiment = mlflow.get_experiment_by_name("Image Captioning")
if experiment is None:
    experiment_id = mlflow.create_experiment("Image Captioning")
else:
    experiment_id = experiment.experiment_id
mlflow.start_run(experiment_id=experiment_id)
mlflow.log_params(training_params)
mlflow.log_artifact("simple_vocab.pkl")


if opt_name == "adam":
        optimizer = torch.optim.Adam(params,lr)
elif opt_name == "sgd":
    optimizer = torch.optim.SGD(params,lr)
elif opt_name == "asgd":
    optimizer = torch.optim.ASGD(params,lr)
elif opt_name == "rprop":
    optimizer = torch.optim.Rprop(params,lr)

if scheduler_name == "logistic":
    scheduler = schedulers.RampUpLogisticDecayScheduler(optimizer, lr, 1e-4, total_step, rampup_period, 1e-5)
if scheduler_name == "cosine":
    scheduler = schedulers.RampUpCosineDecayScheduler(optimizer, lr, 1e-4, total_step, rampup_period, 1e-5)
elif scheduler_name == "cosine_annealing":
    scheduler = schedulers.RampUpCosineAnnealingScheduler(optimizer, lr, 1e-4, rampup_period, 1000,2, 1e-5)
elif scheduler_name == "constant":
    scheduler = schedulers.RampUpScheduler(optimizer, lr, rampup_period, 1e-6)
elif scheduler_name == "steps":
    scheduler = schedulers.RampUpSteps(optimizer, {20000:1e-3, 50000:5e-4, 70000:1e-4, 90000:5e-5}, rampup_period)

acc_loss = 0
best_loss = 0

log_metric_interval = 100

for i_step in tqdm.tqdm(range(1, total_step+1)):

    # Randomly sample a caption length, and sample indices with that length.
    indices = data_loader.dataset.get_train_indices()
    # Create and assign a batch sampler to retrieve a batch with the sampled indices.
    new_sampler = data.sampler.SubsetRandomSampler(indices=indices)
    data_loader.batch_sampler.sampler = new_sampler
    
    # Obtain the batch.
    images, captions = next(iter(data_loader))
    images = transform_train(images)

    # Move batch of images and captions to GPU if CUDA is available.
    images = images.to(device)
    captions = captions.to(device)
    
    # Zero the gradients.
    optimizer.zero_grad()
    
    # Pass the inputs through the CNN-RNN model.
    output = model.compute_gradients(images, captions)
    
    #output = torch.permute(output, (0,2,1))
    
    # Calculate the batch loss.
    # output = output.view(-1, vocab_size)
    # captions = captions.view(-1)
    loss = criterion(output.view(-1,vocab_size), captions.view(-1))
    acc_loss = loss.item()

    # Backward pass.
    loss.backward()

    #torch.nn.utils.clip_grad_norm_(params, 4)
    optimizer.step()
    if scheduler is not None:
        scheduler.step()
        current_lr = scheduler.get_last_lr()
    else:
        current_lr = lr

    if int(i_step%log_metric_interval==log_metric_interval-1):
        stats = {"loss": acc_loss, "lr":current_lr}
        try:
            mlflow.log_metrics(stats, i_step)
        except mlflow.MlflowException as e:
            print(e.message)
    
    if int(i_step%last_every)==last_every-1:
        continue_uploading = True
        while continue_uploading:
            try:
                # mlflow.pytorch.log_model(model,"last",extra_files=["model.py"])
                model.save("last")
                mlflow.log_artifact("last.onnx")
                continue_uploading = False
            except mlflow.MlflowException as e:
                print(e.message)
                time.sleep(5)
    

    # Save the weights.
    if (i_step-1)%save_every == save_every - 1:

        model.eval()

        acc_test_loss = 0
        count = 0
        # Model validation
        with torch.no_grad():

            for images, captions in tqdm.tqdm(data_loader_valid.dataset):

                images = transform_valid(images)
                images = images.unsqueeze(0)
                captions = captions.unsqueeze(0)

                images = images.to(device)
                captions = captions.to(device)

                output = model(images)
                
                if output.shape[1] < captions.shape[1]:
                    captions = captions[:,:output.shape[1]]
                elif output.shape[1] > captions.shape[1]:
                    output = output[:,:captions.shape[1]]
                
                # Calculate the batch loss.
                acc = eval_criterion(output, captions)
                acc_test_loss += acc.item()
                count+=1
            
        acc_test_loss = acc_test_loss/count
        stats = {"jaccard_index_valid": acc_test_loss}
        mlflow.log_metrics(stats, i_step)

        if best_loss < acc_test_loss:
            best_loss = acc_test_loss
            
            continue_uploading = True
            while continue_uploading:
                try:
                    # mlflow.pytorch.log_model(model,"best",extra_files=["model.py"])
                    model.save("best")
                    mlflow.log_artifact("best.onnx")
                    continue_uploading = False
                except mlflow.MlflowException as e:
                    print(e.message)
                    time.sleep(5)
        

        model.train()
            
            
# mlflow.pytorch.log_model(model,"final",extra_files=["model.py"])
model.save("final")
mlflow.log_artifact("final.onnx")

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here. 

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

In [ ]:
# (Optional) TODO: Validate your model.

# The model validation was implemented on the training code in the cell above. It has a comment indicating the start of the validation block.

# Appendix I - Graphics


## Training data

### Metadata

| Start Time  | Duration    | Source Name | Status      |
|-------------|-------------|-------------|-------------|
2024-10-11 20:52:37|	2.1d|	train_model.py|	FINISHED|	

### Parameters

|opt|scheduler|backbone_type| batch_size  | dropout     | embed_size  | hidden_size|num_layers  |lr   |rampup_period|steps        |vocab_size   |vocab_threshold|
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
|adam|steps|efficientnet_v2|	64|	0.4|	512|	512|	1| 0.001|	1000|	100000|	5359|	20|	

### Metrics
|jaccard_index_valid|loss|lr|
|-------------|-------------|-------------|
|0.13942691209820085|	2.0640957355499268|	0.00005|

## Training Loss

<center>
<img src="./images/training_loss.png"/> <br>
Loss per step 
</center>

## Training Learning Rate

<center>
<img src="./images/lr.png"/><br>
Learning rate per step
</center>

## Validation Metric
<center>
<img src="./images/jaccard_index_valid.png"/><br>
Jaccard Index for validation set per step
</center>

# Appendix II - Model Architecture Diagrams

<div>

<div style="float: left; width: 33.333%;">
<center>
<h2> Encoder Architecture </h2> <br>
<img src="./images/test_encoder.onnx.svg"/><br>
</center>
</div>

<div style="display: inline-block; width: 33.333%;">
<center>
<h2> Embedding Architecture </h2> <br>
<img src="./images/test_embed.onnx.svg"/><br>
</center>
</div>

<div style="float: right; width: 33.333%;">
<center>
<h2> Decoder Architecture </h2> <br>
<img src="./images/test_decoder.onnx.svg"/><br>
</center>
</div>

</div>

